# Lesson 1.1 — AWS Global Infrastructure & IAM Fundamentals

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Describe AWS Regions, Availability Zones, and Edge Locations
- Explain the AWS Shared Responsibility Model
- Create IAM users, groups, roles, and policies using the AWS CLI
- Apply the principle of least privilege to IAM permissions

**Exam relevance:** SOA-C03 Domain 4 (Security & Compliance), SAP-C02 Domain 1 (Organizational Complexity)

_Reference: [AWS Global Infrastructure](https://aws.amazon.com/about-aws/global-infrastructure/) | [IAM Documentation](https://docs.aws.amazon.com/iam/latest/userguide/introduction.html)_

## 1. AWS Global Infrastructure

Think of AWS as a worldwide network of data centers organized into three levels:

### Regions
A **Region** is a geographic area containing multiple isolated data center clusters (e.g., `us-east-1` = Northern Virginia, `eu-west-1` = Ireland). When you create an AWS resource, you choose which Region it lives in. Data does NOT automatically replicate across Regions — you must explicitly configure cross-Region replication.

**Why it matters for CloudOps:** You choose Regions for latency, data residency compliance (GDPR, HIPAA), and disaster recovery.

### Availability Zones (AZs)
Each Region has **2–6 Availability Zones** — physically separate data centers within the same Region, connected by low-latency fiber. Think of AZs as isolated failure domains: a fire in one AZ won't affect another.

**Rule of thumb:** Deploy across ≥2 AZs for high availability.

### Edge Locations
**Edge Locations** are CDN endpoints used by **Amazon CloudFront** and **Route 53** to serve content closer to users. There are 400+ edge locations worldwide — many more than Regions.

### Local Zones & Wavelength Zones
- **Local Zones** extend AWS infrastructure closer to major cities for ultra-low latency (e.g., `us-east-1-bos-1` = Boston)
- **Wavelength Zones** embed AWS compute inside telecom 5G networks

---

## 2. The AWS Shared Responsibility Model

This is one of the most important concepts for both exams:

| AWS Responsible For (Security **OF** the Cloud) | You Responsible For (Security **IN** the Cloud) |
|---|---|
| Physical data centers | Your data |
| Hardware and networking | IAM users and permissions |
| Hypervisor | OS patching (on EC2) |
| Managed service infrastructure | Application code |
| Global network infrastructure | Encryption configuration |

**Example:** If you leave an S3 bucket public, that's your responsibility — not AWS's.

In [ ]:
# Lesson 1.1 — Setup
# Verify AWS CLI is installed and configured
import subprocess, json

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
    else:
        print(result.stdout)
    return result.stdout

print("=== AWS CLI Version ===")
run("aws --version")

print("=== Current AWS Identity ===")
run("aws sts get-caller-identity")

## 3. AWS Identity and Access Management (IAM)

**IAM** is AWS's identity and access control service. It is **global** — not region-specific.

### Key IAM Concepts

| Concept | What it is | Analogy |
|---------|-----------|--------|
| **User** | A person or app with long-term credentials | Employee badge |
| **Group** | Collection of users sharing the same permissions | Department team |
| **Role** | Temporary credentials assumed by services or users | Contractor pass |
| **Policy** | JSON document defining allowed/denied actions | Rulebook |

### Policy Anatomy
```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": ["s3:GetObject", "s3:PutObject"],
      "Resource": "arn:aws:s3:::my-bucket/*",
      "Condition": {
        "StringEquals": {"s3:prefix": "uploads/"}
      }
    }
  ]
}
```

### Principle of Least Privilege
Grant only the minimum permissions required to do the job. Start with zero access, add only what's needed.

_Reference: [IAM Best Practices](https://docs.aws.amazon.com/IAM/latest/UserGuide/best-practices.html)_

In [ ]:
# List all available AWS Regions
import boto3, json

ec2 = boto3.client('ec2', region_name='us-east-1')
regions = ec2.describe_regions(AllRegions=False)  # AllRegions=False = only opted-in regions

print(f"Available AWS Regions ({len(regions['Regions'])}):\n")
for r in sorted(regions['Regions'], key=lambda x: x['RegionName']):
    print(f"  {r['RegionName']:25} endpoint: {r['Endpoint']}")

In [ ]:
# List Availability Zones in a specific region
ec2 = boto3.client('ec2', region_name='us-east-1')
azs = ec2.describe_availability_zones(Filters=[{'Name': 'state', 'Values': ['available']}])

print("Availability Zones in us-east-1:")
for az in azs['AvailabilityZones']:
    print(f"  {az['ZoneName']} ({az['State']}) — Zone ID: {az['ZoneId']}")

In [ ]:
# Create an IAM group and attach a managed policy
iam = boto3.client('iam')

GROUP_NAME = 'CloudOpsEngineers'

try:
    response = iam.create_group(GroupName=GROUP_NAME)
    print(f"Created group: {GROUP_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"Group '{GROUP_NAME}' already exists — skipping creation.")

# Attach the AWS managed ReadOnlyAccess policy
iam.attach_group_policy(
    GroupName=GROUP_NAME,
    PolicyArn='arn:aws:iam::aws:policy/ReadOnlyAccess'
)
print(f"Attached ReadOnlyAccess to group: {GROUP_NAME}")

In [ ]:
import json

# Create a custom IAM policy following least privilege
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "CloudWatchReadOnly",
            "Effect": "Allow",
            "Action": [
                "cloudwatch:GetMetricData",
                "cloudwatch:GetMetricStatistics",
                "cloudwatch:ListMetrics",
                "cloudwatch:DescribeAlarms",
                "logs:DescribeLogGroups",
                "logs:DescribeLogStreams",
                "logs:GetLogEvents"
            ],
            "Resource": "*"
        }
    ]
}

POLICY_NAME = 'CloudOpsMonitoringReadOnly'

try:
    response = iam.create_policy(
        PolicyName=POLICY_NAME,
        PolicyDocument=json.dumps(policy_document),
        Description='Read-only access to CloudWatch metrics and logs for CloudOps team'
    )
    policy_arn = response['Policy']['Arn']
    print(f"Created policy: {policy_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    # Get account ID to build the ARN
    account_id = boto3.client('sts').get_caller_identity()['Account']
    policy_arn = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"
    print(f"Policy already exists: {policy_arn}")

In [ ]:
# Create an IAM Role for EC2 to call CloudWatch
# Roles use 'trust policies' to define who can ASSUME the role

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "ec2.amazonaws.com"},  # EC2 can assume this role
            "Action": "sts:AssumeRole"
        }
    ]
}

ROLE_NAME = 'EC2CloudWatchRole'

try:
    role_response = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Allows EC2 instances to write metrics to CloudWatch'
    )
    print(f"Created role: {role_response['Role']['Arn']}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"Role '{ROLE_NAME}' already exists.")

# Attach CloudWatch permissions to the role
iam.attach_role_policy(
    RoleName=ROLE_NAME,
    PolicyArn='arn:aws:iam::aws:policy/CloudWatchAgentServerPolicy'
)
print(f"Attached CloudWatchAgentServerPolicy to role: {ROLE_NAME}")

## Exercise

Complete the following tasks in the cells below:

1. **List all IAM groups** in your account using boto3
2. **Create an IAM policy** that allows `ec2:DescribeInstances` and `ec2:DescribeTags` only
3. **Use the IAM policy simulator concept**: Write a Python function that checks if a given action is in a policy's allowed actions list

In [ ]:
# TODO: List all IAM groups in your account
# Hint: use iam.list_groups()
# Expected output: list of group names

iam = boto3.client('iam')
# YOUR CODE HERE

In [ ]:
# TODO: Create an IAM policy that allows ONLY:
#   - ec2:DescribeInstances
#   - ec2:DescribeTags
# on all resources
# Policy name: 'EC2ReadOnlyTagsPolicy'

ec2_policy = {
    "Version": "2012-10-17",
    "Statement": [
        # YOUR CODE HERE
    ]
}

# YOUR CODE HERE to create the policy

## Cleanup

Run the cell below to remove resources created in this lesson and avoid ongoing charges.

In [ ]:
# Cleanup — run this after completing the lesson
import boto3
iam = boto3.client('iam')
account_id = boto3.client('sts').get_caller_identity()['Account']

# Detach policies from group
try:
    iam.detach_group_policy(GroupName='CloudOpsEngineers', PolicyArn='arn:aws:iam::aws:policy/ReadOnlyAccess')
    iam.delete_group(GroupName='CloudOpsEngineers')
    print("Deleted group: CloudOpsEngineers")
except Exception as e:
    print(f"Group cleanup: {e}")

# Delete custom policy
try:
    iam.delete_policy(PolicyArn=f"arn:aws:iam::{account_id}:policy/CloudOpsMonitoringReadOnly")
    print("Deleted policy: CloudOpsMonitoringReadOnly")
except Exception as e:
    print(f"Policy cleanup: {e}")

# Detach and delete role
try:
    iam.detach_role_policy(RoleName='EC2CloudWatchRole', PolicyArn='arn:aws:iam::aws:policy/CloudWatchAgentServerPolicy')
    iam.delete_role(RoleName='EC2CloudWatchRole')
    print("Deleted role: EC2CloudWatchRole")
except Exception as e:
    print(f"Role cleanup: {e}")

## Summary

In this lesson you:
- Mapped AWS global infrastructure: Regions → AZs → Edge Locations
- Understood the Shared Responsibility Model
- Created IAM groups, custom policies, and roles using boto3
- Applied the principle of least privilege

## Key Exam Points (SOA-C03 / SAP-C02)
- IAM is **global** (not regional)
- Roles use **trust policies** (who can assume) + **permission policies** (what they can do)
- **Explicit Deny** always overrides Allow in IAM
- For EC2 access to AWS services, always use an **IAM Role** (never embed access keys in code)

---

**Next:** [Lesson 1.2 — AWS Well-Architected Framework](lesson-1.2-well-architected.ipynb)